In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import platform
import re
from pathlib import Path

if platform.system() == "Darwin":   # macOS
    plt.rcParams["font.family"] = "AppleGothic"
else:                               # Windows / Linux
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

In [ ]:
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / "Data").exists():
    project_root = project_root.parent

data_path = project_root / "Data" / "ttareung" / "data.parquet"
if not data_path.exists():
    raise FileNotFoundError(f"data.parquet not found: {data_path}")

target_station = {
    "대여소_ID": "ST-71",
    "표시_역명": "225. 앙카라공원 앞",
    "실시간_위도": 37.51758575,
    "실시간_경도": 126.92961884,
}

df = pd.read_parquet(
    data_path,
    columns=[
        "기준_날짜",
        "집계_기준",
        "기준_시간대",
        "시작_대여소_ID",
        "시작_대여소명",
        "종료_대여소_ID",
        "종료_대여소명",
        "전체_건수",
    ],
)
df = df[df["기준_날짜"].between(20240101, 20241231)].copy()
df["시간"] = (df["기준_시간대"].fillna(0).astype(int) // 100).astype(int)

departure_rows = df[(df["시작_대여소_ID"] == target_station["대여소_ID"]) & (df["집계_기준"] == "출발시간")].copy()
arrival_rows = df[(df["종료_대여소_ID"] == target_station["대여소_ID"]) & (df["집계_기준"] == "도착시간")].copy()
reference_rows = df[(df["종료_대여소_ID"] == target_station["대여소_ID"]) & (df["집계_기준"] == "출발시간")].copy()

summary = pd.DataFrame(
    [
        {"구분": "출발 검증용 raw row", "조건": "시작_대여소_ID == ST-71 and 집계_기준 == 출발시간", "row 수": len(departure_rows), "전체_건수 합": float(departure_rows["전체_건수"].sum())},
        {"구분": "도착 검증용 raw row", "조건": "종료_대여소_ID == ST-71 and 집계_기준 == 도착시간", "row 수": len(arrival_rows), "전체_건수 합": float(arrival_rows["전체_건수"].sum())},
        {"구분": "참고용 raw row", "조건": "종료_대여소_ID == ST-71 and 집계_기준 == 출발시간", "row 수": len(reference_rows), "전체_건수 합": float(reference_rows["전체_건수"].sum())},
    ]
)

print("검증 대상")
display(pd.DataFrame([target_station]))

print("요약")
display(summary)

print("ST-71에 매핑된 parquet 내부 종료 대여소명")
display(
    df.loc[df["종료_대여소_ID"] == target_station["대여소_ID"], ["종료_대여소명"]]
    .drop_duplicates()
    .sort_values("종료_대여소명")
    .reset_index(drop=True)
)

print("출발 검증용 raw row")
display(
    departure_rows[
        ["기준_날짜", "기준_시간대", "시간", "시작_대여소_ID", "시작_대여소명", "종료_대여소_ID", "종료_대여소명", "전체_건수"]
    ]
    .sort_values(["기준_날짜", "기준_시간대"])
)

print("도착 검증용 raw row")
display(
    arrival_rows[
        ["기준_날짜", "기준_시간대", "시간", "시작_대여소_ID", "시작_대여소명", "종료_대여소_ID", "종료_대여소명", "전체_건수"]
    ]
    .sort_values(["기준_날짜", "기준_시간대"])
)

print("참고: 같은 종료역(ST-71)을 출발 시각 기준으로 본 raw row")
display(
    reference_rows[
        ["기준_날짜", "기준_시간대", "시간", "시작_대여소_ID", "시작_대여소명", "종료_대여소_ID", "종료_대여소명", "전체_건수"]
    ]
    .sort_values(["기준_날짜", "기준_시간대"])
)

print("시간대별 합계")
hourly_summary = pd.concat(
    [
        departure_rows.groupby("시간", as_index=False)["전체_건수"].sum().assign(구분="출발 검증용"),
        arrival_rows.groupby("시간", as_index=False)["전체_건수"].sum().assign(구분="도착 검증용"),
        reference_rows.groupby("시간", as_index=False)["전체_건수"].sum().assign(구분="참고용(종료역 기준 출발시각)"),
    ],
    ignore_index=True,
)
display(hourly_summary.sort_values(["구분", "시간"]).reset_index(drop=True))